# CS535 Project 2 — IMDb: 1D CNN → DistilBERT → Multi-window CNN

This notebook is the conceptual bridge between image CNNs and Protein Language Models.

## Main learning sequence
1. Load and inspect IMDb.
2. Build a word-level vocabulary.
3. Train a non-pretrained Embedding + 1D CNN baseline.
4. Extend to multi-window 1D CNN.
5. Generate contextual token embeddings with DistilBERT.
6. Train a frozen-DistilBERT + multi-window CNN classifier.
7. Compare metrics and error cases.
8. Map every concept to ESM-2 / protein sequences.

**Research rule:** use validation data for architecture selection. Keep the test set untouched until the final comparison.

In [ ]:
!pip -q install datasets transformers accelerate scikit-learn

In [ ]:
import re, math, random, time, os, sys
from collections import Counter
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, confusion_matrix

SEED=535
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
FAST_MODE=True
print("Device:",DEVICE)

## 1. Load IMDb

Hugging Face IMDb has 25,000 labeled train reviews, 25,000 labeled test reviews, and 50,000 unlabeled reviews.

We split the official training set into our own train/validation subsets. The official test set remains the final independent test.

In [ ]:
imdb=load_dataset("imdb")
print(imdb)

train_raw=imdb["train"].train_test_split(test_size=0.2,seed=SEED,stratify_by_column="label")
train_ds=train_raw["train"]
val_ds=train_raw["test"]
test_ds=imdb["test"]

if FAST_MODE:
    train_ds=train_ds.shuffle(seed=SEED).select(range(6000))
    val_ds=val_ds.shuffle(seed=SEED).select(range(1500))
    test_ds=test_ds.shuffle(seed=SEED).select(range(2000))

print(len(train_ds),len(val_ds),len(test_ds))
print(train_ds[0])

## 2. Baseline vocabulary: no pretrained model yet

This creates a transparent baseline:

`tokens → learned embedding → Conv1D → pooling → classifier`

In [ ]:
TOKEN_RE=re.compile(r"[A-Za-z']+")
def simple_tokenize(text):
    return TOKEN_RE.findall(text.lower())

counter=Counter()
for x in train_ds["text"]:
    counter.update(simple_tokenize(x))

MAX_VOCAB=30000
itos=["<pad>","<unk>"]+[w for w,_ in counter.most_common(MAX_VOCAB-2)]
stoi={w:i for i,w in enumerate(itos)}
PAD,UNK=0,1
print("vocab:",len(itos))

In [ ]:
MAX_LEN=300
def encode_text(text):
    ids=[stoi.get(w,UNK) for w in simple_tokenize(text)[:MAX_LEN]]
    if len(ids)<MAX_LEN: ids += [PAD]*(MAX_LEN-len(ids))
    return torch.tensor(ids,dtype=torch.long)

class IMDBWordDataset(Dataset):
    def __init__(self,hf_ds): self.ds=hf_ds
    def __len__(self): return len(self.ds)
    def __getitem__(self,i):
        x=self.ds[i]
        return encode_text(x["text"]), torch.tensor(float(x["label"]))

def make_word_loaders(bs=64):
    return (
        DataLoader(IMDBWordDataset(train_ds),batch_size=bs,shuffle=True),
        DataLoader(IMDBWordDataset(val_ds),batch_size=bs),
        DataLoader(IMDBWordDataset(test_ds),batch_size=bs)
    )
word_train,word_val,word_test=make_word_loaders()

## 3. Multi-window 1D CNN from scratch

If the embedding dimension is `D`, PyTorch Conv1D expects input as `[batch, channels=D, sequence_length=L]`.

Each convolutional window learns a different local text pattern.

In [ ]:
class TextMCNN(nn.Module):
    def __init__(self,vocab_size,emb_dim=128,windows=(2,3,4,5),filters=64,dropout=0.3):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,emb_dim,padding_idx=0)
        self.convs=nn.ModuleList([nn.Conv1d(emb_dim,filters,k) for k in windows])
        self.dropout=nn.Dropout(dropout)
        self.fc=nn.Linear(filters*len(windows),1)
    def forward(self,ids):
        x=self.embedding(ids).transpose(1,2)
        feats=[F.adaptive_max_pool1d(F.relu(conv(x)),1).squeeze(-1) for conv in self.convs]
        return self.fc(self.dropout(torch.cat(feats,dim=1))).squeeze(1)

word_model=TextMCNN(len(itos)).to(DEVICE)
print(word_model)
print("params:",sum(p.numel() for p in word_model.parameters()))

In [ ]:
def binary_metrics(y,p):
    pred=(p>=0.5).astype(int)
    return {
        "accuracy":accuracy_score(y,pred),
        "precision":precision_score(y,pred,zero_division=0),
        "recall":recall_score(y,pred,zero_division=0),
        "f1":f1_score(y,pred,zero_division=0),
        "roc_auc":roc_auc_score(y,p),
        "pr_auc":average_precision_score(y,p)
    }

def eval_loader(model,loader):
    model.eval(); ys=[]; ps=[]
    with torch.no_grad():
        for xb,yb in loader:
            xb=xb.to(DEVICE); yb=yb.to(DEVICE)
            prob=torch.sigmoid(model(xb))
            ys.extend(yb.cpu().numpy()); ps.extend(prob.cpu().numpy())
    return np.array(ys),np.array(ps)

def fit_binary(model,train_loader,val_loader,epochs=3,lr=1e-3):
    opt=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=lr)
    lossfn=nn.BCEWithLogitsLoss()
    best=None; best_auc=-1; hist=[]
    for ep in range(1,epochs+1):
        model.train(); total=0
        for xb,yb in train_loader:
            xb=xb.to(DEVICE); yb=yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss=lossfn(model(xb),yb); loss.backward(); opt.step()
            total += loss.item()*len(yb)
        y,p=eval_loader(model,val_loader); m=binary_metrics(y,p)
        hist.append({"epoch":ep,"loss":total/len(train_loader.dataset),**m})
        print(hist[-1])
        if m["roc_auc"]>best_auc:
            best_auc=m["roc_auc"]
            best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best)
    return pd.DataFrame(hist)

hist_word=fit_binary(word_model,word_train,word_val,epochs=2 if FAST_MODE else 6)

## 4. Controlled window experiment

Change only the convolution window while keeping the representation and filter count fixed.

In [ ]:
window_results=[]
for w in [2,3,4,5,7]:
    m=TextMCNN(len(itos),windows=(w,),filters=64).to(DEVICE)
    _=fit_binary(m,word_train,word_val,epochs=1 if FAST_MODE else 4)
    y,p=eval_loader(m,word_val)
    window_results.append({"window":w,**binary_metrics(y,p)})
display(pd.DataFrame(window_results))

# Part II — Contextual embeddings with DistilBERT

Now replace the task-trained embedding layer with a pretrained contextual language model.

For teaching speed we freeze DistilBERT and train only the multi-window CNN head.

In [ ]:
MODEL_NAME="distilbert-base-uncased"
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
encoder=AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
for p in encoder.parameters(): p.requires_grad=False
encoder.eval()

BERT_MAX_LEN=192

In [ ]:
class IMDBBertDataset(Dataset):
    def __init__(self,hf_ds): self.ds=hf_ds
    def __len__(self): return len(self.ds)
    def __getitem__(self,i):
        x=self.ds[i]
        enc=tokenizer(x["text"],truncation=True,max_length=BERT_MAX_LEN,padding="max_length",return_tensors="pt")
        return enc["input_ids"].squeeze(0),enc["attention_mask"].squeeze(0),torch.tensor(float(x["label"]))

def bert_loader(ds,bs=16,shuffle=False):
    return DataLoader(IMDBBertDataset(ds),batch_size=bs,shuffle=shuffle)

bert_train=bert_loader(train_ds,16,True)
bert_val=bert_loader(val_ds,32)
bert_test=bert_loader(test_ds,32)

In [ ]:
class FrozenBertMCNN(nn.Module):
    def __init__(self,encoder,hidden=768,windows=(2,3,4,5),filters=64,dropout=0.3):
        super().__init__()
        self.encoder=encoder
        self.convs=nn.ModuleList([nn.Conv1d(hidden,filters,k) for k in windows])
        self.dropout=nn.Dropout(dropout)
        self.fc=nn.Linear(filters*len(windows),1)
    def forward(self,input_ids,attention_mask):
        with torch.no_grad():
            h=self.encoder(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state
        h=h*attention_mask.unsqueeze(-1)
        x=h.transpose(1,2)
        feats=[F.adaptive_max_pool1d(F.relu(c(x)),1).squeeze(-1) for c in self.convs]
        return self.fc(self.dropout(torch.cat(feats,1))).squeeze(1)

bert_mcnn=FrozenBertMCNN(encoder).to(DEVICE)
print("trainable params:",sum(p.numel() for p in bert_mcnn.parameters() if p.requires_grad))

In [ ]:
def eval_bert(model,loader):
    model.eval(); ys=[];ps=[]
    with torch.no_grad():
        for ids,mask,y in loader:
            ids,mask,y=ids.to(DEVICE),mask.to(DEVICE),y.to(DEVICE)
            p=torch.sigmoid(model(ids,mask))
            ys.extend(y.cpu().numpy());ps.extend(p.cpu().numpy())
    return np.array(ys),np.array(ps)

def fit_bert_head(model,train_loader,val_loader,epochs=2,lr=1e-3):
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.AdamW(params,lr=lr); lossfn=nn.BCEWithLogitsLoss()
    best=None;best_auc=-1
    for ep in range(1,epochs+1):
        model.train()
        for ids,mask,y in train_loader:
            ids,mask,y=ids.to(DEVICE),mask.to(DEVICE),y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            loss=lossfn(model(ids,mask),y);loss.backward();opt.step()
        yy,pp=eval_bert(model,val_loader);m=binary_metrics(yy,pp)
        print("epoch",ep,m)
        if m["roc_auc"]>best_auc:
            best_auc=m["roc_auc"];best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    model.load_state_dict(best)

fit_bert_head(bert_mcnn,bert_train,bert_val,epochs=1 if FAST_MODE else 3)

## 5. Final comparison on the untouched test subset

In [ ]:
y1,p1=eval_loader(word_model,word_test)
y2,p2=eval_bert(bert_mcnn,bert_test)
display(pd.DataFrame([
    {"model":"task embedding + MCNN",**binary_metrics(y1,p1)},
    {"model":"frozen DistilBERT + MCNN",**binary_metrics(y2,p2)}
]))

## 6. Error analysis

Inspect high-confidence mistakes. Pay special attention to negation, sarcasm, very long reviews, and mixed sentiment.

In [ ]:
rows=[]
texts=test_ds["text"]
for i,(y,p) in enumerate(zip(y2,p2)):
    pred=int(p>=0.5)
    if pred!=int(y):
        rows.append({"text":texts[i][:500],"true":int(y),"prob_positive":float(p),"confidence":float(max(p,1-p))})
display(pd.DataFrame(rows).sort_values("confidence",ascending=False).head(15))

# Bridge to DeepNeoAG

You now know the entire computational pattern:

```text
text → pretrained LM → token embeddings → multi-window Conv1D → classification
```

Next, replace:
- text with peptide,
- subword token with amino acid,
- DistilBERT with ESM-2 / ProtTrans,
- sentiment with neoantigen function.